In [3]:
import pandas as pd
df = pd.read_csv("../data/raw/chennai_weather.csv")
df.head()


,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,precipitation_probability_max,windspeed_10m_max,relative_humidity_2m_mean,city
0,2022-01-01,27.0,22.8,25.2,8.6,NaN,25.3,81,Chennai
1,2022-01-02,27.9,23.9,25.8,0.9,NaN,20.9,76,Chennai
2,2022-01-03,27.7,23.8,25.5,0.0,NaN,19.7,74,Chennai
3,2022-01-04,27.3,22.5,24.8,0.0,NaN,21.0,73,Chennai
4,2022-01-05,28.0,21.6,24.7,0.1,NaN,14.5,73,Chennai


In [4]:
print(df.shape)
print(df.isna().sum())
df.describe()


(1096, 9)
date                                0
temperature_2m_max                  0
temperature_2m_min                  0
temperature_2m_mean                 0
precipitation_sum                   0
precipitation_probability_max    1096
windspeed_10m_max                   0
relative_humidity_2m_mean           0
city                                0
dtype: int64


,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,precipitation_probability_max,windspeed_10m_max,relative_humidity_2m_mean
count,1096.000000,1096.000000,1096.000000,1096.000000,0.0,1096.000000,1096.000000
mean,31.833394,25.073723,28.056843,4.267427,NaN,18.765602,75.725365
std,3.069329,2.134920,2.155471,12.452897,NaN,4.670351,7.816990
min,24.100000,18.200000,21.900000,0.000000,NaN,9.400000,47.000000
25%,29.200000,23.800000,26.200000,0.000000,NaN,15.500000,72.000000
50%,31.900000,25.200000,28.200000,0.500000,NaN,18.300000,77.000000
75%,34.100000,26.600000,29.800000,3.600000,NaN,21.700000,81.000000
max,41.000000,30.400000,34.800000,255.800000,NaN,53.100000,94.000000


In [5]:
import pandas as pd
df = pd.read_csv("../data/raw/chennai_weather.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.drop(columns=["precipitation_probability_max"])
df = df.sort_values("date").reset_index(drop=True)
df["rained"] = (df["precipitation_sum"] > 1).astype(int)
print(df["rained"].value_counts())
df.head()

rained
0    645
1    451
Name: count, dtype: int64


,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,windspeed_10m_max,relative_humidity_2m_mean,city,rained
0,2022-01-01,27.0,22.8,25.2,8.6,25.3,81,Chennai,1
1,2022-01-02,27.9,23.9,25.8,0.9,20.9,76,Chennai,0
2,2022-01-03,27.7,23.8,25.5,0.0,19.7,74,Chennai,0
3,2022-01-04,27.3,22.5,24.8,0.0,21.0,73,Chennai,0
4,2022-01-05,28.0,21.6,24.7,0.1,14.5,73,Chennai,0


In [6]:
df["temp_mean_lag1"] = df["temperature_2m_mean"].shift(1)
df["rain_lag1"] = df["precipitation_sum"].shift(1)
df["humidity_lag1"] = df["relative_humidity_2m_mean"].shift(1)
df["temp_mean_roll7"] = df["temperature_2m_mean"].rolling(7).mean()
df["rain_sum_roll7"] = df["precipitation_sum"].rolling(7).sum()
import numpy as np
df["day_of_year"] = df["date"].dt.dayofyear
df["season_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["season_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)
df = df.dropna().reset_index(drop=True)
print(df.shape)
df.head()

(1090, 17)


,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,windspeed_10m_max,relative_humidity_2m_mean,city,rained,temp_mean_lag1,rain_lag1,humidity_lag1,temp_mean_roll7,rain_sum_roll7,day_of_year,season_sin,season_cos
0,2022-01-07,27.3,22.2,25.1,2.5,16.6,78,Chennai,1,25.0,0.1,77.0,25.157143,12.2,7,0.120208,0.992749
1,2022-01-08,27.8,22.5,25.4,0.4,17.4,76,Chennai,0,25.1,2.5,78.0,25.185714,4.0,8,0.137279,0.990532
2,2022-01-09,28.1,22.3,25.3,0.0,12.7,77,Chennai,0,25.4,0.4,76.0,25.114286,3.1,9,0.154309,0.988023
3,2022-01-10,28.5,21.9,25.6,0.0,12.5,76,Chennai,0,25.3,0.0,77.0,25.128571,3.1,10,0.171293,0.985220
4,2022-01-11,28.7,23.2,25.9,0.2,15.6,78,Chennai,0,25.6,0.0,76.0,25.285714,3.3,11,0.188227,0.982126


In [7]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/chennai_weather_features.csv", index=False)
print("Saved cleaned dataset with features")

Saved cleaned dataset with features


In [14]:
# =========================================================
# FEATURES AND TARGETS
# =========================================================
feature_cols = [
    "temp_mean_lag1",
    "rain_lag1",
    "humidity_lag1",
    "temp_mean_roll7",
    "rain_sum_roll7",
    "season_sin",
    "season_cos",
    "windspeed_10m_max"
]

# Input features
X = df[feature_cols]

# Prediction targets
y_temp = df["temperature_2m_mean"]
y_rain = df["rained"]
y_humidity = df["relative_humidity_2m_mean"]

# Time-series split: first 80% for training, last 20% for testing
split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_temp_train = y_temp.iloc[:split_idx]
y_temp_test = y_temp.iloc[split_idx:]

y_rain_train = y_rain.iloc[:split_idx]
y_rain_test = y_rain.iloc[split_idx:]

y_humidity_train = y_humidity.iloc[:split_idx]
y_humidity_test = y_humidity.iloc[split_idx:]

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 872, Test size: 218


In [15]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTemperature target:")
print("Train:", y_temp_train.shape)
print("Test:", y_temp_test.shape)

print("\nRain target:")
print("Train:", y_rain_train.shape)
print("Test:", y_rain_test.shape)

print("\nHumidity target:")
print("Train:", y_humidity_train.shape)
print("Test:", y_humidity_test.shape)

print("\nFeatures being used:")
print(feature_cols)

X_train shape: (872, 8)
X_test shape: (218, 8)

Temperature target:
Train: (872,)
Test: (218,)

Rain target:
Train: (872,)
Test: (218,)

Humidity target:
Train: (872,)
Test: (218,)

Features being used:
['temp_mean_lag1', 'rain_lag1', 'humidity_lag1', 'temp_mean_roll7', 'rain_sum_roll7', 'season_sin', 'season_cos', 'windspeed_10m_max']


In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd


# =========================================================
# TEMPERATURE MODELS
# =========================================================

temp_models = {

    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        random_state=42
    )
}


temp_results = []

for name, model in temp_models.items():

    model.fit(X_train, y_temp_train)

    preds = model.predict(X_test)

    mae = mean_absolute_error(y_temp_test, preds)

    rmse = np.sqrt(
        mean_squared_error(y_temp_test, preds)
    )

    temp_results.append({
        "Model": name,
        "MAE (°C)": round(mae, 2),
        "RMSE (°C)": round(rmse, 2)
    })


temp_results_df = pd.DataFrame(temp_results)


print("========================================")
print("TEMPERATURE MODEL RESULTS")
print("========================================")

print(temp_results_df)


# =========================================================
# HUMIDITY MODELS
# =========================================================

humidity_models = {

    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        random_state=42
    )
}


humidity_results = []

for name, model in humidity_models.items():

    model.fit(X_train, y_humidity_train)

    preds = model.predict(X_test)

    mae = mean_absolute_error(
        y_humidity_test,
        preds
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_humidity_test,
            preds
        )
    )

    humidity_results.append({
        "Model": name,
        "MAE (%)": round(mae, 2),
        "RMSE (%)": round(rmse, 2)
    })


humidity_results_df = pd.DataFrame(
    humidity_results
)


print("\n========================================")
print("HUMIDITY MODEL RESULTS")
print("========================================")

print(humidity_results_df)

TEMPERATURE MODEL RESULTS
               Model  MAE (°C)  RMSE (°C)
0  Linear Regression      0.51       0.68
1      Random Forest      0.53       0.70
2            XGBoost      0.58       0.75

HUMIDITY MODEL RESULTS
               Model  MAE (%)  RMSE (%)
0  Linear Regression     3.43      4.38
1      Random Forest     3.61      4.53
2            XGBoost     4.01      4.99


In [17]:
# =========================================================
# RAIN CLASSIFICATION MODELS
# =========================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


rain_models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        random_state=42,
        eval_metric="logloss"
    )
}


rain_results = []


for name, model in rain_models.items():

    model.fit(
        X_train,
        y_rain_train
    )

    preds = model.predict(X_test)

    accuracy = accuracy_score(
        y_rain_test,
        preds
    )

    precision = precision_score(
        y_rain_test,
        preds,
        zero_division=0
    )

    recall = recall_score(
        y_rain_test,
        preds,
        zero_division=0
    )

    f1 = f1_score(
        y_rain_test,
        preds,
        zero_division=0
    )

    rain_results.append({
        "Model": name,
        "Accuracy": round(accuracy, 2),
        "Precision": round(precision, 2),
        "Recall": round(recall, 2),
        "F1 Score": round(f1, 2)
    })


rain_results_df = pd.DataFrame(
    rain_results
)


print("========================================")
print("RAIN MODEL RESULTS")
print("========================================")

print(rain_results_df)

RAIN MODEL RESULTS
                 Model  Accuracy  Precision  Recall  F1 Score
0  Logistic Regression      0.63       0.69    0.67      0.68
1        Random Forest      0.67       0.71    0.75      0.73
2              XGBoost      0.67       0.70    0.77      0.73


In [18]:
# =========================================================
# SAVE FINAL MODELS
# =========================================================

import os
import joblib

# Create models folder if it doesn't exist
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)


# ---------------------------------------------------------
# TRAIN FINAL TEMPERATURE MODEL
# ---------------------------------------------------------

final_temp_model = LinearRegression()

final_temp_model.fit(
    X_train,
    y_temp_train
)


# ---------------------------------------------------------
# TRAIN FINAL HUMIDITY MODEL
# ---------------------------------------------------------

final_humidity_model = LinearRegression()

final_humidity_model.fit(
    X_train,
    y_humidity_train
)


# ---------------------------------------------------------
# TRAIN FINAL RAIN MODEL
# ---------------------------------------------------------

final_rain_model = XGBClassifier(
    n_estimators=200,
    random_state=42,
    eval_metric="logloss"
)

final_rain_model.fit(
    X_train,
    y_rain_train
)


# ---------------------------------------------------------
# SAVE MODELS
# ---------------------------------------------------------

joblib.dump(
    final_temp_model,
    os.path.join(models_dir, "temp_model.pkl")
)

joblib.dump(
    final_rain_model,
    os.path.join(models_dir, "rain_model.pkl")
)

joblib.dump(
    final_humidity_model,
    os.path.join(models_dir, "humidity_model.pkl")
)

joblib.dump(
    feature_cols,
    os.path.join(models_dir, "feature_cols.pkl")
)


print("========================================")
print("MODELS SAVED SUCCESSFULLY")
print("========================================")

print("Temperature model → models/temp_model.pkl")
print("Rain model        → models/rain_model.pkl")
print("Humidity model    → models/humidity_model.pkl")
print("Feature columns   → models/feature_cols.pkl")

MODELS SAVED SUCCESSFULLY
Temperature model → models/temp_model.pkl
Rain model        → models/rain_model.pkl
Humidity model    → models/humidity_model.pkl
Feature columns   → models/feature_cols.pkl


In [10]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
rain_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, random_state=42)
}
rain_results = []
for name, model in rain_models.items():
    model.fit(X_train, y_rain_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_rain_test, preds)
    rain_results.append({"Model": name, "Accuracy": round(acc, 3)})
rain_results_df = pd.DataFrame(rain_results)
print(rain_results_df)

                 Model  Accuracy
0  Logistic Regression     0.775
1        Random Forest     0.725
2              XGBoost     0.711


In [11]:
baseline_temp_preds = X_test["temp_mean_lag1"]
baseline_mae = mean_absolute_error(y_temp_test, baseline_temp_preds)
print(f"Naive persistence baseline MAE: {baseline_mae:.2f}°C")
majority_class = y_rain_train.mode()[0]
baseline_rain_preds = [majority_class] * len(y_rain_test)
baseline_acc = accuracy_score(y_rain_test, baseline_rain_preds)
print(f"Naive majority-class baseline accuracy: {baseline_acc:.3f}")

Naive persistence baseline MAE: 0.54°C
Naive majority-class baseline accuracy: 0.413


In [12]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

# Retrain best models on ALL data (train+test) for the final saved version
best_temp_model = LinearRegression()
best_temp_model.fit(X, y_temp)
joblib.dump(best_temp_model, "../models/temp_model.pkl")

best_rain_model = LogisticRegression(max_iter=1000)
best_rain_model.fit(X, y_rain)
joblib.dump(best_rain_model, "../models/rain_model.pkl")

# Also save the feature column order — the API will need this later
joblib.dump(feature_cols, "../models/feature_cols.pkl")

print("Models saved!")

Models saved!
